In [ ]:
import sys
from pathlib import Path

In [ ]:
src_path = Path("../src").resolve()
sys.path.append(str(src_path))

In [ ]:
from api.events.models import EventModel
from api.db.session import engine
from sqlmodel import Session, select
from sqlalchemy import literal_column

In [ ]:
with Session(engine) as session:
    query = select(EventModel).order_by(EventModel.updated_at.asc()).limit(10)
    compiled_query = query.compile(compile_kwargs={"literal_binds": True})
    print(compiled_query)
    print("")
    print(str(query))
    results = session.exec(query).fetchall()
    pprint(results)

In [ ]:
from sqlmodel import func 
from datetime import datetime, timedelta, timezone
from pprint import pprint


with Session(engine) as session:
    interval = literal_column("'1 minute'")
    bucket = func.time_bucket(interval, EventModel.created_at)
    pages = ['/about', '/contact', '/pages', '/pricing']
    start = datetime.now(timezone.utc) - timedelta(hours=1)
    finish = datetime.now(timezone.utc) + timedelta(hours=1)
    query = (
        select(
            bucket,
            EventModel.page,
            func.count()
        )
        .where(EventModel.created_at > start,
                EventModel.created_at <= finish,
                EventModel.page == "/about")
        .group_by(
            bucket,
            EventModel.page,
        )
        .order_by(
            bucket,
            EventModel.page,
        )
    )
    compiled_query = query.compile(compile_kwargs={"literal_binds": True})
    # print(compiled_query)
    results = session.exec(query).fetchall()
    pprint(results)